In [0]:
pip install pytrends

In [0]:
# If needed (run once on a normal cluster):
# %pip install pytrends

import json
import datetime as dt
import pandas as pd
from typing import Dict, Any
from pytrends.request import TrendReq

# ---------------- Config ----------------
CATALOG = "canada_business"
SCHEMA  = "bronze"
BRONZE_SUBDIR = "google_trends_raw"

GEO = "CA"
KEYWORDS = ["buy laptop", "TV deals", "gaming console", "electronics sale"]
TIMEFRAME = "today 5-y"   # backfill-friendly; later you can change to "today 3-m" for daily

# ---------------- Helpers (same pattern as your FRED ingest) ----------------
def run_ts_utc() -> str:
    return dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")

def resolve_schema_root_location(catalog: str, schema: str) -> str:
    df = spark.sql(f"DESCRIBE SCHEMA EXTENDED {catalog}.{schema}").toPandas()
    rows = df.loc[df["database_description_item"] == "RootLocation", "database_description_value"].values
    if len(rows) == 0 or not rows[0]:
        raise RuntimeError(f"RootLocation not found for {catalog}.{schema}")
    return str(rows[0])

def join_uri(base_uri: str, child: str) -> str:
    return base_uri.rstrip("/") + "/" + child.strip("/")

# ---------------- Paths ----------------
root_location = resolve_schema_root_location(CATALOG, SCHEMA)
bronze_base = join_uri(root_location, BRONZE_SUBDIR)

data_base = bronze_base.rstrip("/") + "/data"
manifest_base = bronze_base.rstrip("/") + "/manifests"

dbutils.fs.mkdirs(data_base)
dbutils.fs.mkdirs(manifest_base)

# ---------------- Fetch from Google Trends ----------------
ts = run_ts_utc()

pytrends = TrendReq(hl="en-CA", tz=0)
pytrends.build_payload(KEYWORDS, timeframe=TIMEFRAME, geo=GEO)

df = pytrends.interest_over_time().reset_index()
if "isPartial" in df.columns:
    df = df.drop(columns=["isPartial"])

manifest: Dict[str, Any] = {
    "run_ts": ts,
    "geo": GEO,
    "timeframe": TIMEFRAME,
    "keywords": KEYWORDS,
    "files": []
}

# ---------------- Write raw JSON files (1 file per keyword) ----------------
for kw in KEYWORDS:
    kw_dir = kw.replace(" ", "_").lower()
    out_dir = join_uri(data_base, kw_dir)
    dbutils.fs.mkdirs(out_dir)

    out_file = join_uri(out_dir, f"trend_{ts}.json")

    payload = {
        "keyword": kw,
        "geo": GEO,
        "timeframe": TIMEFRAME,
        "run_ts": ts,
        "observations": [
            {"date": str(r["date"].date()), "value": float(r[kw]) if pd.notna(r[kw]) else None}
            for _, r in df.iterrows()
        ]
    }

    dbutils.fs.put(out_file, json.dumps(payload, indent=2), overwrite=False)

    manifest["files"].append({
        "keyword": kw,
        "rows": len(payload["observations"]),
        "raw_file": out_file
    })

manifest_path = join_uri(manifest_base, f"run_manifest_{ts}.json")
dbutils.fs.put(manifest_path, json.dumps(manifest, indent=2), overwrite=False)

print("✅ Google Trends ingest complete")
print("Bronze base:", bronze_base)
print("Manifest:", manifest_path)
for f in manifest["files"]:
    print(f"✅ {f['keyword']} -> {f['rows']} rows -> {f['raw_file']}")